##creating gold tables

In [0]:
orders = spark.table(
    "brazilian_ecommerce_public.silver.orders_silver"
)

customers = spark.table(
    "brazilian_ecommerce_public.silver.customers_silver"
)

payments = spark.table(
    "brazilian_ecommerce_public.silver.order_payments_silver"
)

In [0]:
fact_sales = (
    orders
    .join(customers, "customer_id", "left")
    .join(payments, "order_id", "left")
)

display(fact_sales)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_sequential,payment_type,payment_installments,payment_value
e4de6d53ecff736bc68804b0b6e9f635,9f6618c17568ac301465fe7ad056c674,delivered,2017-10-16T14:56:50.000Z,2017-10-17T03:49:34.000Z,2017-10-27T22:14:21.000Z,2017-11-08T21:25:24.000Z,2017-11-21T00:00:00.000Z,e3bcfea9bab07b492391664fc1ffc28a,44180,antonio cardoso,BA,1,boleto,1,231.12
f4471dae8c482f51aa1826cd9f5d4433,167b9485947ed0a354a3f8dad04eb199,delivered,2018-07-05T18:40:47.000Z,2018-07-05T18:55:15.000Z,2018-07-10T15:10:00.000Z,2018-07-11T21:16:47.000Z,2018-07-19T00:00:00.000Z,548a09978548d2e347d494793e34c797,6462,barueri,SP,1,credit_card,8,232.14
0fc34e566d3273325c404c113f4a8b2a,ae9cc596d22235c547acce001f90fc33,delivered,2017-11-02T14:51:11.000Z,2017-11-02T15:10:35.000Z,2017-11-06T20:28:43.000Z,2017-12-01T18:06:48.000Z,2017-11-27T00:00:00.000Z,930dfe46b364af81db0fb4828bce74c9,74675,goiania,GO,1,credit_card,2,205.07
0acdae809654a2de1235d18a12cf73f3,3180b241917d07e27bd46aac80ded579,delivered,2018-06-14T11:29:22.000Z,2018-06-14T12:19:40.000Z,2018-06-15T14:22:00.000Z,2018-06-19T21:46:19.000Z,2018-07-05T00:00:00.000Z,06bf11f58f4a6b5ab7dd1623404bf672,30120,belo horizonte,MG,1,debit_card,1,66.07
4f392bdcc6b33f1b56ae0fd844343bbb,d12dfe84678bf6a057d218e2ee623154,delivered,2017-05-12T22:50:56.000Z,2017-05-13T07:22:56.000Z,2017-05-16T09:17:35.000Z,2017-05-17T12:08:39.000Z,2017-06-05T00:00:00.000Z,5898ff704517d3dcae16ca666b5f225a,13403,piracicaba,SP,1,credit_card,1,93.48
b84c178bc4308555f74ed9cdf3a38193,22a1be4f281a33941e88bc790cfb60a6,delivered,2018-01-25T11:03:37.000Z,2018-01-25T11:12:33.000Z,2018-01-29T19:24:48.000Z,2018-02-07T18:36:54.000Z,2018-02-22T00:00:00.000Z,51601e58354758c975c5e878a11b4352,89202,joinville,SC,1,credit_card,1,138.6
e35b67f3b9778766f35d9d6e11af1761,df9c907a19d1c599e071af85fcb04f92,delivered,2018-04-03T22:54:22.000Z,2018-04-03T23:10:13.000Z,2018-04-07T01:18:30.000Z,2018-04-10T16:20:58.000Z,2018-04-25T00:00:00.000Z,6c1e77450766afe18d4a99135d322d89,13073,campinas,SP,1,credit_card,1,171.55
0bf1392e4d1b5bb968191f48c30e5628,b8ec04c8a20aa6136b207e84ec12593f,delivered,2017-11-26T15:50:34.000Z,2017-11-28T16:03:01.000Z,2017-11-29T20:22:50.000Z,2017-12-15T16:32:05.000Z,2017-12-15T00:00:00.000Z,374ad33a8b3ec3e6fac9cbceaeb727ec,15540,alvares florence,SP,1,credit_card,3,129.79
e80d8758f771bccd842da8b6fc2837ca,f11e9b6d22751ce7f65db87b615fea21,delivered,2018-03-19T17:53:45.000Z,2018-03-19T18:15:33.000Z,2018-03-20T18:08:28.000Z,2018-03-24T15:03:53.000Z,2018-03-29T00:00:00.000Z,c325d123fdea8b29701a0a93c7955ca0,18108,sorocaba,SP,1,credit_card,3,156.63
4dd077c76e72e73742c4af14ebfbd4d1,5184df07acf834cd8a00cda25c376ebd,delivered,2018-06-07T08:57:05.000Z,2018-06-07T09:15:24.000Z,2018-06-08T14:44:00.000Z,2018-06-15T00:22:21.000Z,2018-07-13T00:00:00.000Z,d15efa6fcfd5909767d5c9de1608af29,1313,sao paulo,SP,1,credit_card,2,151.62


In [0]:
fact_sales.write \
.mode("overwrite") \
.saveAsTable(
    "brazilian_ecommerce_public.gold.fact_sales"
)

# another gold table 

In [0]:
from pyspark.sql.functions import sum

customer_revenue = (
    fact_sales
    .groupBy("customer_unique_id")
    .agg(
        sum("payment_value")
        .alias("total_revenue")
    )
)

display(customer_revenue)

customer_unique_id,total_revenue
e3bcfea9bab07b492391664fc1ffc28a,231.12
18bbfd7c77ff481bac5ae34fe5b8cc52,62.74
5493b0c34dc1481b67a6365a6c3ff174,38.14
0047f3e16441284d757a8963344f6c59,76.18
0fdc0d21e1983e8af4d399e17671f76d,112.2
6c1e77450766afe18d4a99135d322d89,171.55
e559839364c5291151976b2c69daa5fd,93.54
cde21887e86a254eab4799aec212b7b7,675.01
1fe1ebaae0912c827a35e8a93c1c5ce7,26.44
64be1085f459ad4720cd416c386d3e51,260.27


In [0]:
customer_revenue.write \
.mode("overwrite") \
.saveAsTable(
    "brazilian_ecommerce_public.gold.customer_revenue"
)